## 4. Interpretation & Findings

**Key Observations:**
- **Weak Correlation**: Most stocks show weak correlations (|r| < 0.5), suggesting sentiment alone is not a reliable price predictor.
- **Data Sparsity**: Limited overlapping dates between news and trading days reduce sample sizes, impacting statistical power.
- **Time Lag Effects**: News sentiment may have a lagged impact on prices (reaction occurs 1–5 days later), not immediately.
- **Confounding Factors**: Earnings announcements, Fed policy, and macro events dominate returns; individual news sentiment is secondary.

**Recommendations:**
1. **Expand time windows**: Look 1–5 days ahead to capture lagged sentiment effects.
2. **Aggregate sentiment**: Use rolling 5-day average sentiment to smooth noise.
3. **Combine signals**: Pair sentiment with technical indicators and volume for stronger predictive power.
4. **Regime analysis**: Test sentiment correlation separately during bull/bear markets.

In [ ]:
# Focus on GOOG (highest correlation strength)
stock = 'GOOG'
stock_df = load_stock(f'{stock}.csv')
stock_df['Daily_Return'] = compute_returns(stock_df, 'Close')
if 'Date' in stock_df.columns:
    stock_df['Date'] = pd.to_datetime(stock_df['Date'])
    stock_df['Date_Only'] = stock_df['Date'].dt.date
else:
    print(f"Cannot process {stock}")
    stock_df = None

if stock_df is not None:
    sentiment_goog = daily_sentiment[daily_sentiment['stock'] == stock].copy()
    sentiment_goog['Date_Only'] = pd.to_datetime(sentiment_goog['date'], utc=True, format='mixed', errors='coerce').dt.date

    merged_goog = sentiment_goog.merge(stock_df[['Date_Only', 'Daily_Return']], on='Date_Only', how='inner')

    if len(merged_goog) > 0:
        # Create scatter plot
        fig, ax = plt.subplots(figsize=(12, 7))

        scatter = ax.scatter(merged_goog['avg_sentiment'], merged_goog['Daily_Return'], 
                             s=100, alpha=0.6, c=merged_goog['avg_sentiment'], cmap='RdYlGn', edgecolors='black')

        # Add trend line
        z = np.polyfit(merged_goog['avg_sentiment'], merged_goog['Daily_Return'], 1)
        p = np.poly1d(z)
        x_line = np.linspace(merged_goog['avg_sentiment'].min(), merged_goog['avg_sentiment'].max(), 100)
        ax.plot(x_line, p(x_line), "r--", linewidth=2, alpha=0.8, label=f"Trend: y={z[0]:.2f}x+{z[1]:.2f}")

        ax.axhline(0, color='gray', linestyle='-', linewidth=0.8, alpha=0.5)
        ax.axvline(0, color='gray', linestyle='-', linewidth=0.8, alpha=0.5)

        ax.set_xlabel('Daily Average Sentiment (Compound Score)', fontsize=11)
        ax.set_ylabel('Daily Return (%)', fontsize=11)
        
        if len(correlations) > 2:
            ax.set_title(f'{stock}: Sentiment vs. Daily Returns (r={correlations[2]["Correlation"]:.4f}, p={correlations[2]["P_Value"]:.4f})', fontsize=12)
        else:
            ax.set_title(f'{stock}: Sentiment vs. Daily Returns', fontsize=12)
        ax.legend()
        ax.grid(alpha=0.3)

        plt.colorbar(scatter, ax=ax, label='Sentiment')
        plt.tight_layout()
        plt.show()

        # Summary statistics for merged data
        print(f"\n{stock} Merged Data Statistics (n={len(merged_goog)}):")
        print(f"Sentiment  - Mean: {merged_goog['avg_sentiment'].mean():.3f}, Std: {merged_goog['avg_sentiment'].std():.3f}")
        print(f"Daily Ret. - Mean: {merged_goog['Daily_Return'].mean():.3f}%, Std: {merged_goog['Daily_Return'].std():.3f}%")
    else:
        print(f"No overlapping data for {stock}")

## 3. Sentiment vs. Returns Scatter Plot (GOOG Example)

Visualize the sentiment-return relationship for the stock with the strongest data coverage.

In [ ]:
# Compute correlations for each stock
correlations = []

for stock in sorted(daily_sentiment['stock'].unique()):
    # Load stock prices
    stock_file = f'{stock}.csv'
    stock_df = load_stock(stock_file)
    if len(stock_df) == 0:
        print(f"{stock}: Cannot load stock data")
        continue
    stock_df['Daily_Return'] = compute_returns(stock_df, 'Close')
    
    # Normalize dates
    if 'Date' in stock_df.columns:
        stock_df['Date'] = pd.to_datetime(stock_df['Date'])
        stock_df['Date_Only'] = stock_df['Date'].dt.date
    else:
        continue
    
    sentiment_df = daily_sentiment[daily_sentiment['stock'] == stock].copy()
    sentiment_df['Date_Only'] = pd.to_datetime(sentiment_df['date'], utc=True, format='mixed', errors='coerce').dt.date
    
    # Merge on date
    merged = sentiment_df.merge(stock_df[['Date_Only', 'Daily_Return']], on='Date_Only', how='inner')
    
    if len(merged) > 2:
        corr, pval = pearsonr(merged['avg_sentiment'], merged['Daily_Return'])
        correlations.append({
            'Stock': stock,
            'N_Observations': len(merged),
            'Correlation': corr,
            'P_Value': pval,
            'Significant': 'Yes' if pval < 0.05 else 'No'
        })
        print(f"{stock}: r={corr:.4f}, p={pval:.4f}, n={len(merged)}")
    else:
        print(f"{stock}: Insufficient data (n={len(merged)})")

corr_df = pd.DataFrame(correlations)
print("\n" + "="*60)
print("CORRELATION SUMMARY")
print("="*60)
print(corr_df.to_string(index=False))

## 2. Sentiment-Return Correlation

Compute Pearson correlation between daily sentiment and daily stock returns. A positive correlation suggests that positive sentiment predicts price appreciation; negative correlation suggests an inverse relationship or contrarian effect.

In [ ]:
# Sentiment statistics by stock
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
daily_sentiment.boxplot(column='avg_sentiment', by='stock', ax=axes[0])
axes[0].set_xlabel('Stock')
axes[0].set_ylabel('Average Sentiment (Compound Score)')
axes[0].set_title('Sentiment Distribution by Stock')
axes[0].axhline(0, color='red', linestyle='--', linewidth=1, alpha=0.7)
plt.sca(axes[0])
plt.xticks(rotation=0)

# Bar plot of mean sentiment
sentiment_by_stock = daily_sentiment.groupby('stock')['avg_sentiment'].mean().sort_values(ascending=False)
sentiment_by_stock.plot(kind='bar', ax=axes[1], color=['green' if x > 0 else 'red' for x in sentiment_by_stock.values])
axes[1].set_xlabel('Stock')
axes[1].set_ylabel('Mean Sentiment')
axes[1].set_title('Average Sentiment by Stock')
axes[1].axhline(0, color='black', linestyle='-', linewidth=1)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("Sentiment Statistics by Stock:")
for stock in sorted(daily_sentiment['stock'].unique()):
    stock_sentiment = daily_sentiment[daily_sentiment['stock'] == stock]['avg_sentiment']
    print(f"\n{stock}:")
    print(f"  Mean: {stock_sentiment.mean():.3f}")
    print(f"  Std:  {stock_sentiment.std():.3f}")
    print(f"  Min:  {stock_sentiment.min():.3f}")
    print(f"  Max:  {stock_sentiment.max():.3f}")

## 1. Sentiment Distribution by Stock

Understanding the sentiment landscape helps identify publication bias and market sentiment trends.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_news, load_stock
from src.sentiment import apply_vader, aggregate_daily_sentiment
from src.indicators import compute_returns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 7)

print("Loading and processing data...")
# Load and process news
news = load_news()
news = apply_vader(news, 'headline')
daily_sentiment = aggregate_daily_sentiment(news, 'date', 'stock')
print(f"Aggregated {len(news)} headlines into {len(daily_sentiment)} daily sentiment records")

# Sentiment-Return Correlation Analysis

This notebook analyzes the relationship between news sentiment and stock returns, testing the hypothesis that positive sentiment predicts price appreciation.